# 04 — Primary sparse Graphical LASSO partial-correlation network

Estimates the primary conditional-dependency network via Graphical LASSO. Model selection minimizes EBIC(gamma=0.5) subject to a pre-specified sparse-interpretability constraint of network density ≤ 0.20 (Methods 2.8). This is the primary, prespecified analytical model of the study (n = 1286, 29 direct biomarkers).

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json
import numpy as np, pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()
Z = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_z.csv')
cols = primary
p = len(cols)

model, alpha_table, selected_alpha = select_glasso_ebic(Z, max_density=0.20)

out_dir = PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso'
out_dir.mkdir(parents=True, exist_ok=True)
alpha_table.to_csv(out_dir / 'primary_alpha_selection_ebic.csv', index=False)

prec = model.precision_
D = np.sqrt(np.diag(prec))
P = -prec / np.outer(D, D)
np.fill_diagonal(P, 1)

edges = edge_df_from_partial(P, cols)
edges['source_domain'] = edges.source.map(DOM)
edges['target_domain'] = edges.target.map(DOM)
edges['cross_domain'] = edges.source_domain != edges.target_domain
edges.to_csv(out_dir / 'primary_graphical_lasso_partial_edges.csv', index=False)
pd.DataFrame(P, index=cols, columns=cols).to_csv(out_dir / 'primary_partial_correlation_matrix.csv')

summary = {
    'n_participants': int(len(Z)),
    'n_primary_direct_biomarkers': int(len(cols)),
    'selection_method': 'EBIC gamma=0.50 with pre-specified sparse interpretability constraint density <= 0.20',
    'selected_alpha': float(selected_alpha),
    'n_edges': int(len(edges)),
    'network_density': float(len(edges) / (p * (p - 1) / 2)),
    'n_cross_domain_edges': int(edges.cross_domain.sum()),
    'primary_network_note': 'This is the primary network. Unconstrained EBIC and alternative gamma values are reported only as sensitivity because they produced a dense network.',
}
(out_dir / 'primary_graphical_lasso_model_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


### Network figure

In [ ]:
plt.figure(figsize=(11, 9))
G = nx.Graph()
for f in cols:
    G.add_node(f, domain=DOM[f])
for _, r in edges.iterrows():
    G.add_edge(r.source, r.target, weight=r.weight, abs_weight=r.abs_weight)
pos = nx.spring_layout(G, seed=SEED, k=0.55)
widths = [max(0.5, 4 * G[u][v]['abs_weight']) for u, v in G.edges]
nx.draw_networkx_edges(G, pos, width=widths, alpha=.45)
nx.draw_networkx_nodes(G, pos, node_size=650)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.axis('off'); plt.tight_layout()
plt.savefig(out_dir / 'Figure_primary_partial_network.png', dpi=200)
plt.show()
